### PyIQA

In [ ]:
# !pip install pyiqa
conda activate your_env_name
pip install pyiqa


: 

In [1]:
# # Use the conda environment where you installed pyiqa
# import pyiqa
# import torch

# # list all available metrics
# print(pyiqa.list_models())

# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# # create metric with default setting
# iqa_metric = pyiqa.create_metric('laion_aes', device=device)

# # check if lower better or higher better
# print(iqa_metric.lower_better)

# # example for iqa score inference
# # Tensor inputs, img_tensor_x/y: (N, 3, H, W), RGB, 0 ~ 1
# # score_fr = iqa_metric(img_tensor_x, img_tensor_y)

# # img path as inputs.
# score_fr = iqa_metric('./images/a_girl_9.png')
# print(score_fr)

# # For FID metric, use directory or precomputed statistics as inputs
# # refer to clean-fid for more details: https://github.com/GaParmar/clean-fid
# # fid_metric = pyiqa.create_metric('fid')
# # score = fid_metric('./ResultsCalibra/dist_dir/', './ResultsCalibra/ref_dir')
# # score = fid_metric('./ResultsCalibra/dist_dir/', dataset_name="FFHQ", dataset_res=1024, dataset_split="trainval70k")

In [3]:
# pip install --force-reinstall --upgrade torchvision==0.19.1+cu124 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# pip install pyiqa

In [3]:
# pip install --upgrade --force-reinstall numpy scipy

In [2]:
# Applying multiple IQA models to an image to derive results (Code was tested on Google Colab so might need to install some libraries)
# !pip install pyiqa - https://pypi.org/project/pyiqa/

# BRIQSUE - https://live.ece.utexas.edu/publications/2011/am_asilomar_2011.pdf / https://zenodo.org/records/11104461 / https://learnopencv.com/image-quality-assessment-brisque/ / https://www.mathworks.com/help/images/ref/brisque.html
# NIQE - https://github.com/Wangyanouc/Test/blob/master/Papers/IQA/Making%20a%20Completely%20Blind%20Image%20Quality%20Analyzer(NIQE)%20.pdf/ https://www.mathworks.com/help/images/ref/niqe.html
# PaQ-2-PiQ - https://baidut.github.io/PaQ-2-PiQ/#download-zone / https://github.com/baidut/paq2piq
# MANIQA - https://arxiv.org/abs/2204.08958

import pyiqa
import torch
from PIL import Image
from torchvision import transforms

import numpy as np
# Resolving dtype issues relating to version compatability
np.float_ = np.float64
np.complex_ = np.complex128

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load and preprocess image
def load_image(image_path):
    img = Image.open(image_path).convert('RGB')
    transform = transforms.ToTensor()  # Converts to (C, H, W) with range [0, 1]
    return transform(img).unsqueeze(0).to(device)  # Shape: (1, 3, H, W)

# Path to your image
image_path = 'images/a car_1 - Copy.png' #'/content/low-quality-image-generator-v0-1flc72ny8cvb1.webp'
img_tensor = load_image(image_path)

# Create all three metrics
paq2piq = pyiqa.create_metric('paq2piq').to(device)
niqe = pyiqa.create_metric('niqe').to(device)
brisque = pyiqa.create_metric('brisque').to(device)
maniqa = pyiqa.create_metric('maniqa').to(device)

# Get scores
score_paq2piq = paq2piq(img_tensor).item()  # Higher is better
score_niqe = niqe(img_tensor).item()        # Lower is better
score_brisque = brisque(img_tensor).item()  # Lower is better
score_maniqa = maniqa(img_tensor).item()    # Higher is better

# Print results
print(f"📷 Image: {image_path}")
print(f"✅ PaQ-2-PiQ Score:  {score_paq2piq:.4f}  (higher = better)")
print(f"✅ NIQE Score:       {score_niqe:.4f}     (lower = better)")
print(f"✅ BRISQUE Score:    {score_brisque:.4f}  (lower = better)")
print(f"✅ MANIQA Score:     {score_maniqa:.4f}  (higher = better)")



Using device: cuda
Loading pretrained model PAQ2PIQ from C:\Users\User\.cache\torch\hub\pyiqa\P2P_RoIPoolModel-fit.10.bs.120-ca69882e.pth
Loading pretrained model MANIQA from C:\Users\User\.cache\torch\hub\pyiqa\ckpt_koniq10k.pt
📷 Image: images/a car_1 - Copy.png
✅ PaQ-2-PiQ Score:  72.5081  (higher = better)
✅ NIQE Score:       2.3155     (lower = better)
✅ BRISQUE Score:    19.5795  (lower = better)
✅ MANIQA Score:     0.3332  (higher = better)


In [ ]:
# Test the above and other similar models using PLCC (Pearson Linear Correlation Coefficient) & SROCC/SRCC (Spearman Rank Correlation Coefficient) on benchmaks like LIVE, TID2013, CSIQ, and KonIQ-10k to ensure they actually perform well 
# as was done in the paper: "Image Quality Assessment With Compressed Sampling" which can be found here: https://arxiv.org/pdf/2404.17170 or in the Papers directory of this repository.

# Legacy Datasets - LIVE IQA, TID-2008, TID-2013

# New In-the-wild Datasets - CLIVE, KonIQ-10k

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
import pyiqa
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error

# --- Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- Paths ---
tid2008_folder = "IQA_benchmark_datasets\\tid2013"  # replace with your local path
images_folder = os.path.join(tid2008_folder, "distorted_images")
mos_file = os.path.join(tid2008_folder, "mos_with_names.txt")  # contains filenames and MOS

# --- Load MOS ---
mos_data = []
with open(mos_file, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            score, filename = parts
            mos_data.append((filename, float(score)))

# --- Load IQA models ---
# Resolving dtype issues relating to version compatability
np.float_ = np.float64
np.complex_ = np.complex128

# paq2piq = pyiqa.create_metric('paq2piq').to(device)
# niqe = pyiqa.create_metric('niqe').to(device)
# brisque = pyiqa.create_metric('brisque').to(device)
maniqa = pyiqa.create_metric('maniqa-pipal').to(device)
higher_better = not maniqa.lower_better

# --- Image preprocessing ---
transform = transforms.Compose([transforms.ToTensor()])

def load_image_tensor(image_path):
    img = Image.open(image_path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

# --- Evaluate all images ---
pred_paq2piq, pred_niqe, pred_brisque, pred_maniqa = [], [], [], []

for idx, (filename, mos) in enumerate(mos_data):
    img_path = os.path.join(images_folder, filename)
    img_tensor = load_image_tensor(img_path)
    
    with torch.no_grad():
        # pred_paq2piq.append(paq2piq(img_tensor).item())
        # pred_niqe.append(niqe(img_tensor).item())
        # pred_brisque.append(brisque(img_tensor).item())
        pred_maniqa.append(maniqa(img_tensor).item())
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx+1}/{len(mos_data)} images...")

# --- Convert lists to arrays ---
# pred_paq2piq = np.array(pred_paq2piq)
# pred_niqe = np.array(pred_niqe)
# pred_brisque = np.array(pred_brisque)
pred_maniqa = np.array(pred_maniqa)
mos_scores = np.array([score for _, score in mos_data])

# --- Compute SRCC, PLCC, RMSE ---
def compute_metrics(pred, target, higher_better=True):
    if not higher_better:
        pred = -pred  # flip for lower-is-better metrics
    srcc = spearmanr(pred, target).correlation
    plcc = pearsonr(pred, target)[0]
    rmse = np.sqrt(mean_squared_error(pred, target))
    return srcc, plcc, rmse

metrics = {
    # 'PaQ-2-PiQ': compute_metrics(pred_paq2piq, mos_scores, higher_better=True),
    # 'NIQE': compute_metrics(pred_niqe, mos_scores, higher_better=False),
    # 'BRISQUE': compute_metrics(pred_brisque, mos_scores, higher_better=False),
    'MANIQA': compute_metrics(pred_maniqa, mos_scores, higher_better=higher_better),
}

for model, (srcc, plcc, rmse) in metrics.items():
    print(f"{model}: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")


Using device: cuda
Loading pretrained model MANIQA from C:\Users\User\.cache\torch\hub\pyiqa\MANIQA_PIPAL-ae6d356b.pth


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy\lib\site-packages\timm\models\vision_transformer.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  x = F.scaled_dot_product_attention(


Processed 100/3000 images...
Processed 200/3000 images...
Processed 300/3000 images...
Processed 400/3000 images...
Processed 500/3000 images...
Processed 600/3000 images...
Processed 700/3000 images...
Processed 800/3000 images...
Processed 900/3000 images...
Processed 1000/3000 images...
Processed 1100/3000 images...
Processed 1200/3000 images...
Processed 1300/3000 images...
Processed 1400/3000 images...
Processed 1500/3000 images...
Processed 1600/3000 images...
Processed 1700/3000 images...
Processed 1800/3000 images...
Processed 1900/3000 images...
Processed 2000/3000 images...
Processed 2100/3000 images...
Processed 2200/3000 images...
Processed 2300/3000 images...
Processed 2400/3000 images...
Processed 2500/3000 images...
Processed 2600/3000 images...
Processed 2700/3000 images...
Processed 2800/3000 images...
Processed 2900/3000 images...
Processed 3000/3000 images...
MANIQA: SRCC=0.6013, PLCC=0.6981, RMSE=4.0112


To-Do: Identify Model Training Datasets, Replicate Paper Results, Identify Min/Max Model Scores

MODEL | TRAINED ON | (Paper) TESTED ON | MY RESULTS
MANIQA: | PIPAL | TID2013 - SROCC(0.619) / PLCC(0.704) | TID2013 - SRCC(0.6013) PLCC(0.6981) 


| **Model** | **Trained On** | **Tested On** | **(Paper) Results** | **Replicated Results** |
|-----------|----------------|------------------------|----------------------|------------------|
| MANIQA    | PIPAL          | TID2013                | SRCC = 0.619 / PLCC = 0.704 | SRCC = 0.6013 / PLCC = 0.6981 |


In [19]:
# # --- Get ranking indices from existing arrays ---
# def get_ranks_from_array(arr, higher_better=True):
#     if not higher_better:
#         arr = -arr  # flip for lower-is-better metrics
#     return np.argsort(-arr)  # descending order (highest first)

# # Official MOS ranking
# mos_ranks = get_ranks_from_array(mos_scores, higher_better=True)

# # Model rankings
# ranks_paq2piq = get_ranks_from_array(pred_paq2piq, higher_better=True)
# ranks_niqe = get_ranks_from_array(pred_niqe, higher_better=False)   # lower better
# ranks_brisque = get_ranks_from_array(pred_brisque, higher_better=False) # lower better
# ranks_maniqa = get_ranks_from_array(pred_maniqa, higher_better=True)

# # --- Function to print top-N ranked images ---
# def print_top_ranks(name, ranks, mos_data, top_n=10):
#     print(f"\nTop-{top_n} images according to {name}:")
#     for i in range(top_n):
#         idx = ranks[i]
#         filename, mos = mos_data[idx]
#         print(f"{i+1:2d}. {filename} (MOS={mos})")

# # Official MOS
# print_top_ranks("Official MOS", mos_ranks, mos_data)

# # Model rankings
# print_top_ranks("PaQ-2-PiQ", ranks_paq2piq, mos_data)
# print_top_ranks("NIQE", ranks_niqe, mos_data)
# print_top_ranks("BRISQUE", ranks_brisque, mos_data)
# print_top_ranks("MANIQA", ranks_maniqa, mos_data)


In [ ]:
from scipy.stats import pearsonr, spearmanr

x = y = [1, 2, 3, 4, 5]
# y = [2, 4, 6, 8, 10]
r, p_value = pearsonr(x, y)
print(r)

rho, p_value = spearmanr(x, y)
print(rho) 

1.0
0.9999999999999999


array([73.26839447, 73.31803894, 72.95680237, ..., 74.89868927,
       73.86989594, 72.90370941], shape=(3000,))

In [15]:
from pyiqa import get_dataset_info, load_dataset
import matplotlib.pyplot as plt

# list all available datasets
print(get_dataset_info().keys())

# load dataset with default options
# LIVE = controlled lab distortions
# LIVEM = synthetic multiple distortions
# LIVEC = real-world distortions from phones/cameras
LIVE_dataset = load_dataset('live', data_root='./IQA_benchmark_datasets', force_download=False)
CSIQ_dataset = load_dataset('csiq', data_root='./IQA_benchmark_datasets', force_download=False)
KONIQ10K_dataset = load_dataset('koniq10k', data_root='./IQA_benchmark_datasets', force_download=False)
TID2013_dataset = load_dataset('tid2013', data_root='./IQA_benchmark_datasets', force_download=False)

# print(f'Loaded dataset, len={len(dataset)}, {dataset[0].keys()}')

# for x in dataset:
#     img = x['img']  # tensor (C, H, W)
#     if hasattr(img, "permute"):  # PyTorch tensor
#         img = img.permute(1, 2, 0).numpy()  # (H, W, C)
#     else:  # already numpy array
#         img = img.transpose(1, 2, 0)

#     plt.imshow(img)
#     plt.axis("off")
#     plt.show()
#     print("MOS:", x['mos_label'])



# for x in dataset:
#     plt.imshow(x['img'])   
#     # print(x['img'].shape)
#     # print(x['ref_img'])
#     print(x['mos_label'])
#     # print(x['img_path'])
#     # print(x['ref_img'])

# # split_ratio: train/test/val
# dataset = load_dataset('csiq', data_root='./datasets', force_download=False, split_index=1, split_ratio='622', phase='test')
# print(f'Loaded dataset, len={len(dataset)}, {dataset[0].keys()}')
# print(dataset[0]['img'].shape)

# # or use dataset options
# dataset_opts = {
#   'split_index': 1,
#   'split_ratio': '622',
#   'phase': 'test',
#   'augment': {
#       'resize': 256,
#       'center_crop': 224,
#   }
# }
# dataset = load_dataset('csiq', data_root='./datasets', force_download=False, dataset_opts=dataset_opts)
# print(f'Loaded dataset, len={len(dataset)}, {dataset[0].keys()}')
# print(dataset[0]['img'].shape)

dict_keys(['csiq', 'tid2008', 'tid2013', 'live', 'livem', 'livec', 'koniq10k', 'koniq10k-1024', 'koniq10k++', 'kadid10k', 'spaq', 'ava', 'pipal', 'flive', 'pieapp', 'bapps', 'gfiqa', 'cgfiqa'])
Loading dataset live from ./IQA_benchmark_datasets ...
Loading dataset csiq from ./IQA_benchmark_datasets ...
Loading dataset koniq10k from ./IQA_benchmark_datasets ...
Loading dataset tid2013 from ./IQA_benchmark_datasets ...


In [22]:
for y in [LIVE_dataset, CSIQ_dataset, KONIQ10K_dataset, TID2013_dataset]:
    print(len(y))
    for x in y:
        print(x.keys())
        break

779
dict_keys(['img', 'ref_img', 'mos_label', 'img_path', 'ref_img_path'])
866
dict_keys(['img', 'ref_img', 'mos_label', 'img_path', 'ref_img_path'])
2015
dict_keys(['img', 'mos_label', 'img_path'])
3000
dict_keys(['img', 'ref_img', 'mos_label', 'img_path', 'ref_img_path'])


In [4]:
# import os
# import urllib.request
# import zipfile
# import scipy.io
# import numpy as np
# from PIL import Image
# import torch
# from torchvision import transforms
# import pyiqa
# from scipy.stats import spearmanr, pearsonr
# from sklearn.metrics import mean_squared_error

# # --- Setup ---
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f"Using device: {device}")

# # --- Download and extract TID2013 ---
# tid_url = 'http://www.ponomarenko.info/tid2013/TID2013.zip'
# tid_zip_path = 'TID2013.zip'
# tid_folder = 'TID2013'

# if not os.path.exists(tid_folder):
#     print("Downloading TID2013 dataset...")
#     urllib.request.urlretrieve(tid_url, tid_zip_path)
#     print("Extracting...")
#     with zipfile.ZipFile(tid_zip_path, 'r') as zip_ref:
#         zip_ref.extractall(tid_folder)
#     print("Done.")

# # --- Load MOS scores ---
# mos_mat = scipy.io.loadmat(os.path.join(tid_folder, 'mos_subjective_scores.mat'))
# mos = mos_mat['MOS_mean'].flatten()  # shape (3000,)

# # --- Load images list ---
# ref_images_folder = os.path.join(tid_folder, 'distorted_images')  # distorted images folder
# image_files = sorted([f for f in os.listdir(ref_images_folder) if f.endswith('.bmp')])

# # --- Load IQA models ---
# paq2piq = pyiqa.create_metric('paq2piq').to(device)
# niqe = pyiqa.create_metric('niqe').to(device)
# brisque = pyiqa.create_metric('brisque').to(device)
# maniqa = pyiqa.create_metric('maniqa').to(device)

# # --- Image preprocessing ---
# transform = transforms.Compose([transforms.ToTensor()])

# def load_image_tensor(image_path):
#     img = Image.open(image_path).convert('RGB')
#     return transform(img).unsqueeze(0).to(device)

# # --- Evaluate all images ---
# pred_paq2piq = []
# pred_niqe = []
# pred_brisque = []
# pred_maniqa = []

# for idx, img_name in enumerate(image_files):
#     img_path = os.path.join(ref_images_folder, img_name)
#     img_tensor = load_image_tensor(img_path)
    
#     with torch.no_grad():
#         pred_paq2piq.append(paq2piq(img_tensor).item())
#         pred_niqe.append(niqe(img_tensor).item())
#         pred_brisque.append(brisque(img_tensor).item())
#         pred_maniqa.append(maniqa(img_tensor).item())
    
#     if (idx+1) % 100 == 0:
#         print(f"Processed {idx+1}/{len(image_files)} images...")

# # --- Convert lists to arrays ---
# pred_paq2piq = np.array(pred_paq2piq)
# pred_niqe = np.array(pred_niqe)
# pred_brisque = np.array(pred_brisque)
# pred_maniqa = np.array(pred_maniqa)
# mos = np.array(mos)

# # --- Compute SRCC, PLCC, RMSE ---
# def compute_metrics(pred, target, higher_better=True):
#     if not higher_better:
#         pred = -pred  # flip for lower-is-better metrics
#     srcc = spearmanr(pred, target).correlation
#     plcc = pearsonr(pred, target)[0]
#     rmse = np.sqrt(mean_squared_error(pred, target))
#     return srcc, plcc, rmse

# metrics = {
#     'PaQ-2-PiQ': compute_metrics(pred_paq2piq, mos, higher_better=True),
#     'NIQE': compute_metrics(pred_niqe, mos, higher_better=False),
#     'BRISQUE': compute_metrics(pred_brisque, mos, higher_better=False),
#     'MANIQA': compute_metrics(pred_maniqa, mos, higher_better=True),
# }

# for model, (srcc, plcc, rmse) in metrics.items():
#     print(f"{model}: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")


In [32]:
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error
from scipy.optimize import curve_fit

# --- Setup device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

# --- Common transforms ---
transform = transforms.Compose([transforms.ToTensor()])

# --- Utility functions ---
def load_image_tensor(img_path, transform=transform, device=device):
    """Load a single image as a tensor on the specified device."""
    img = Image.open(img_path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

def compute_metrics(pred, target, higher_better=True):
    """Compute SRCC, PLCC, RMSE."""
    if not higher_better:
        pred = -pred
    srcc = spearmanr(pred, target).correlation
    plcc = pearsonr(pred, target)[0]
    rmse = np.sqrt(mean_squared_error(pred, target))
    return srcc, plcc, rmse

# --- Five-parameter logistic function ---
def logistic5(x, beta1, beta2, beta3, beta4, beta5):
    return beta1 * (0.5 - 1 / (1 + np.exp(beta2 * (x - beta3)))) + beta4 * x + beta5

# --- Fit logistic function ---
def nonlinear_regression(pred_scores, mos_scores):
    # Initial guess for parameters
    beta_init = [1, 1, np.mean(pred_scores), 0, np.mean(mos_scores)]
    
    # Fit curve
    popt, _ = curve_fit(logistic5, pred_scores, mos_scores, p0=beta_init, maxfev=10000)
    
    # Apply mapping
    pred_mapped = logistic5(pred_scores, *popt)
    return pred_mapped

# --- Modular evaluation function ---
def evaluate_iqa_model(model, dataset, batch_size=1, device=device):
    """
    Evaluate any IQA model from `pyiqa` on any dataset in the `load_dataset` structure.
    
    Args:
        model      : a PyTorch IQA model (pyiqa.create_metric(...))
        dataset    : a dataset from `load_dataset` (any split)
        batch_size : number of images to process per batch
        device     : 'cuda' or 'cpu'
    
    Returns:
        srcc, plcc, rmse, pred_scores, mos_scores
    """
    model.to(device).eval()
    pred_scores = []
    mos_scores = []

    use_amp = (device == 'cuda')

    for i in range(0, len(dataset), batch_size):
        print(f"Processing batch {i // batch_size + 1}")
        batch = dataset[i:i+batch_size]

        # Load batch images
        imgs_batch = torch.cat([load_image_tensor(
            item.get('img_path', item.get('img')), transform=transform, device=device
        ) for item in batch], dim=0)

        # Collect MOS labels
        mos_batch = np.array([item['mos_label'] for item in batch])
        mos_scores.extend(mos_batch)

        # Forward pass
        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs_batch)
            else:
                outputs = model(imgs_batch)

        pred_scores.extend(outputs.cpu().numpy())

        if (i + batch_size) % 50 == 0 or (i + batch_size) >= len(dataset):
            print(f"Processed {min(i + batch_size, len(dataset))}/{len(dataset)} images")

    pred_scores = np.array(pred_scores)
    mos_scores = np.array(mos_scores)

    # Map predicted scores to MOS scores
    pred_scores_mapped = nonlinear_regression(pred_scores, mos_scores)

    # Deriving if score is higher better
    higher_better = not model.lower_better

    srcc, plcc, rmse = compute_metrics(pred_scores, mos_scores, higher_better=higher_better)
    print(f"Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    srcc, plcc, rmse = compute_metrics(pred_scores_mapped, mos_scores, higher_better=higher_better)
    print(f"Mapped Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    return srcc, plcc, rmse, pred_scores, mos_scores

Using device: cuda


In [4]:
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error
from scipy.optimize import curve_fit
import torch
from torch.utils.data import DataLoader
from scipy.special import expit
import pyiqa

# --- Setup device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

# --- Common transforms ---
transform = transforms.Compose([transforms.ToTensor()])

# --- Utility functions ---
def load_image_tensor(img_path, transform=transform, device=device):
    """Load a single image as a tensor on the specified device."""
    img = Image.open(img_path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

def compute_metrics(pred, target, higher_better=True):
    """Compute SRCC, PLCC, RMSE."""
    if not higher_better:
        pred = -pred
    srcc = float(spearmanr(pred, target).correlation)
    plcc = float(pearsonr(pred, target)[0])
    rmse = float(np.sqrt(mean_squared_error(pred, target)))
    return srcc, plcc, rmse

# --- Five-parameter logistic function ---
def logistic5(x, beta1, beta2, beta3, beta4, beta5):
    return beta1 * (0.5 - expit(-beta2 * (x - beta3))) + beta4 * x + beta5
    # return beta1 * (0.5 - 1 / (1 + expit(beta2 * (x - beta3)))) + (beta4 * x) + beta5

# --- Fit logistic function ---
def nonlinear_regression(pred_scores, mos_scores):
    # Ensure flat 1-D float arrays
    pred_scores = np.asarray(pred_scores, dtype=np.float64).ravel()
    mos_scores = np.asarray(mos_scores, dtype=np.float64).ravel()

    # Initial guess for parameters
    beta_init = [1, 1, np.mean(pred_scores), 0, np.mean(mos_scores)]
    
    print("pred_scores shape:", pred_scores.shape, "dtype:", pred_scores.dtype)
    print("mos_scores shape:", mos_scores.shape, "dtype:", mos_scores.dtype)
    print("Sample pred_scores:", pred_scores[:5])
    print("Sample mos_scores:", mos_scores[:5])


    # Fit curve
    popt, _ = curve_fit(logistic5, pred_scores, mos_scores, p0=beta_init, maxfev=10000)
    
    # Apply mapping
    pred_mapped = logistic5(pred_scores, *popt)
    return pred_mapped

def collate_fn(batch):
    # batch is a list of dicts: [{'img': tensor, 'mos_label': float}, ...]
    imgs = torch.stack([item['img'] for item in batch], dim=0)
    mos = torch.tensor([item['mos_label'] for item in batch], dtype=torch.float32)
    return imgs, mos

# --- Modular evaluation function ---
def evaluate_iqa_model(model, dataset, batch_size=1, device=device):
    """
    Evaluate any IQA model from `pyiqa` on any dataset in the `load_dataset` structure.
    
    Args:
        model      : a PyTorch IQA model (pyiqa.create_metric(...))
        dataset    : a dataset from `load_dataset` (any split)
        batch_size : number of images to process per batch
        device     : 'cuda' or 'cpu'
    
    Returns:
        srcc, plcc, rmse, pred_scores, mos_scores
    """
    model.to(device).eval()
    pred_scores = []
    mos_scores = []

    use_amp = (device == 'cuda')

    # Create DataLoader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    for batch_idx, batch in enumerate(dataloader, start=1):
        # print(f"Processing batch {batch_idx}/{len(dataloader)}")
        # batch is a tuple: (images, labels)
        imgs_batch, mos_batch = batch

        # Move images to device
        imgs_batch = imgs_batch.to(device)
        mos_batch = mos_batch.numpy()
        mos_scores.extend(mos_batch)

        # Forward pass
        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs_batch)
            else:
                outputs = model(imgs_batch)

        pred_scores.extend(outputs.cpu().numpy())

        if batch_idx % 50 == 0 or batch_idx == len(dataloader):
            print(f"Processed {min(batch_idx * batch_size, len(dataset))}/{len(dataset)} images")

    # pred_scores = np.array([float(x) for x in pred_scores], dtype=np.float64)

    pred_scores = np.array([x.item() if isinstance(x, torch.Tensor) else x for x in pred_scores], dtype=np.float64).ravel()
    mos_scores = np.array([float(x) for x in mos_scores], dtype=np.float64)

    # Map predicted scores to MOS scores
    pred_scores_mapped = nonlinear_regression(pred_scores, mos_scores)

    # print(pred_scores)
    # print(mos_scores)

    # Deriving if score is higher better
    higher_better = not model.lower_better
    print(f"Higher is better: {higher_better}")

    srcc, plcc, rmse = compute_metrics(pred_scores, mos_scores, higher_better=higher_better)
    print(f"Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    srcc, plcc, rmse = compute_metrics(pred_scores_mapped, mos_scores, higher_better=higher_better)
    print(f"Mapped Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    return srcc, plcc, rmse, pred_scores, mos_scores

Using device: cuda


In [37]:
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error
from scipy.optimize import curve_fit
import torch
from torch.utils.data import DataLoader
from scipy.special import expit

# --- Setup device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

# --- Common transforms ---
transform = transforms.Compose([transforms.ToTensor()])

# --- Utility functions ---
def load_image_tensor(img_path, transform=transform, device=device):
    """Load a single image as a tensor on the specified device."""
    img = Image.open(img_path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

def compute_metrics(pred, target, higher_better=True):
    """Compute SRCC, PLCC, RMSE."""
    if not higher_better:
        pred = -pred
    srcc = float(spearmanr(pred, target).correlation)
    plcc = float(pearsonr(pred, target)[0])
    rmse = float(np.sqrt(mean_squared_error(pred, target)))
    return srcc, plcc, rmse

# --- Five-parameter logistic function ---
def logistic5(x, beta1, beta2, beta3, beta4, beta5):
    return beta1 * (0.5 - expit(-beta2 * (x - beta3))) + beta4 * x + beta5 #IDENTIFY WHERE THIS IS FROM
    # return beta1 * (0.5 - 1 / (1 + expit(beta2 * (x - beta3)))) + (beta4 * x) + beta5 -> https://ieeexplore-ieee-org.ejournals.um.edu.mt/stamp/stamp.jsp?tp=&arnumber=1709988

# --- Fit logistic function ---
def nonlinear_regression(pred_scores, mos_scores):
    # Ensure flat 1-D float arrays
    pred_scores = np.asarray(pred_scores, dtype=np.float64).ravel()
    mos_scores = np.asarray(mos_scores, dtype=np.float64).ravel()

    # Initial guess for parameters - Look into the reasoning behind this
    beta_init = [
        np.max(mos_scores) - np.min(mos_scores), # beta1: Range of MOS values
        0.1,                                     # beta2: A small positive value for steepness
        np.mean(pred_scores),                    # beta3: Average of predicted scores (inflection point)
        0,                                       # beta4: Initial slope of the linear part
        np.min(mos_scores)                       # beta5: Starting vertical offset
    ]

    # Fit curve
    popt, _ = curve_fit(logistic5, pred_scores, mos_scores, p0=beta_init, maxfev=100000)
    
    # Apply mapping
    pred_mapped = logistic5(pred_scores, *popt)
    return pred_mapped

def collate_fn(batch):
    # batch is a list of dicts: [{'img': tensor, 'mos_label': float}, ...]
    imgs = torch.stack([item['img'] for item in batch], dim=0)
    mos = torch.tensor([item['mos_label'] for item in batch], dtype=torch.float32)
    return imgs, mos

# --- Modular evaluation function ---
def evaluate_iqa_model(model, dataset, batch_size=1, device=device):
    """
    Evaluate any IQA model from `pyiqa` on any dataset in the `load_dataset` structure.
    
    Args:
        model      : a PyTorch IQA model (pyiqa.create_metric(...))
        dataset    : a dataset from `load_dataset` (any split)
        batch_size : number of images to process per batch
        device     : 'cuda' or 'cpu'
    
    Returns:
        srcc, plcc, rmse, pred_scores, mos_scores
    """
    model.to(device).eval()
    pred_scores = []
    mos_scores = []

    use_amp = (device == 'cuda')

    # Create DataLoader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    for batch_idx, batch in enumerate(dataloader, start=1):
        # batch is a tuple: (images, labels)
        imgs_batch, mos_batch = batch

        # Move images to device
        imgs_batch = imgs_batch.to(device)
        mos_batch = mos_batch.numpy()
        mos_scores.extend(mos_batch)

        # Forward pass
        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs_batch)
            else:
                outputs = model(imgs_batch)

        # --- Normalize the outputs ---
        # Step 1: Handle scalar outputs (e.g., NIQE)
        if outputs.dim() == 0:
            outputs = outputs.unsqueeze(0)

        # Step 2: Handle 2D outputs (e.g., Paq2Piq, MANIQA)
        # This converts a tensor of shape (1, 1) to (1,)
        elif outputs.dim() == 2 and outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)

        # Step 3: Standardize the data type to float32
        outputs = outputs.to(torch.float32)
        # --- End of normalization ---

        pred_scores.extend(outputs.cpu().numpy())

        if batch_idx % 50 == 0 or batch_idx == len(dataloader):
            print(f"Processed {min(batch_idx * batch_size, len(dataset))}/{len(dataset)} images")

    pred_scores = np.array([float(x) for x in pred_scores], dtype=np.float64).ravel()

    # pred_scores = np.array([x.item() if isinstance(x, torch.Tensor) else x for x in pred_scores], dtype=np.float64)
    mos_scores = np.array([float(x) for x in mos_scores], dtype=np.float64)

    # Map predicted scores to MOS scores
    pred_scores_mapped = nonlinear_regression(pred_scores, mos_scores)

    print(f'pred_scores: {pred_scores}')
    print(f'pred_scores_mapped: {pred_scores_mapped}')
    print(f'mos_scores: {mos_scores}')

    # Deriving if score is higher better
    higher_better = not model.lower_better
    # print(f"Higher is better: {higher_better}")

    srcc, plcc, rmse = compute_metrics(pred_scores, mos_scores, higher_better=higher_better)
    print(f"Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    srcc, plcc, rmse = compute_metrics(pred_scores_mapped, mos_scores, higher_better=higher_better)
    print(f"Mapped Results: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")

    return srcc, plcc, rmse, pred_scores, mos_scores

Using device: cuda


In [ ]:
import pyiqa
from pyiqa import load_dataset
from torch.utils.data import Subset

# Resolving dtype issues relating to version compatability
np.float_ = np.float64
np.complex_ = np.complex128

model = pyiqa.create_metric('maniqa-pipal')
dataset =  load_dataset('tid2013', data_root='./IQA_benchmark_datasets', force_download=False)

# limit = 100
# dataset = Subset(dataset, range(limit))

evaluate_iqa_model(model, dataset, batch_size=1, device=device)

Loading pretrained model MANIQA from C:\Users\User\.cache\torch\hub\pyiqa\MANIQA_PIPAL-ae6d356b.pth
Loading dataset tid2013 from ./IQA_benchmark_datasets ...


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv - Copy\lib\site-packages\timm\models\vision_transformer.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  x = F.scaled_dot_product_attention(


Processed 50/3000 images
Processed 100/3000 images
Processed 150/3000 images
Processed 200/3000 images
Processed 250/3000 images
Processed 300/3000 images
Processed 350/3000 images
Processed 400/3000 images
Processed 450/3000 images
Processed 500/3000 images
Processed 550/3000 images
Processed 600/3000 images
Processed 650/3000 images
Processed 700/3000 images
Processed 750/3000 images
Processed 800/3000 images
Processed 850/3000 images
Processed 900/3000 images
Processed 950/3000 images
Processed 1000/3000 images
Processed 1050/3000 images
Processed 1100/3000 images
Processed 1150/3000 images
Processed 1200/3000 images
Processed 1250/3000 images
Processed 1300/3000 images
Processed 1350/3000 images
Processed 1400/3000 images
Processed 1450/3000 images
Processed 1500/3000 images
Processed 1550/3000 images
Processed 1600/3000 images
Processed 1650/3000 images
Processed 1700/3000 images
Processed 1750/3000 images
Processed 1800/3000 images
Processed 1850/3000 images
Processed 1900/3000 i

C:\Users\User\AppData\Local\Temp\ipykernel_27864\2087918753.py:113: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred_scores = np.array([float(x) for x in pred_scores], dtype=np.float64).ravel()


Higher is better: True
Results: SRCC=0.6013, PLCC=0.6981, RMSE=4.0112
Mapped Results: SRCC=0.6013, PLCC=0.7040, RMSE=0.8804


(0.6013199048822588,
 0.7039808923785391,
 0.8804393732308899,
 array([0.76214457, 0.74995357, 0.72787964, ..., 0.70916331, 0.67674416,
        0.55524802], shape=(3000,)),
 array([5.51428986, 5.56757021, 4.94443989, ..., 4.62857008, 4.02778006,
        2.77143002], shape=(3000,)))

In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import torch

# def evaluate_model_on_multiple_datasets_parallel(model, datasets_dict, batch_size=1, device=device, max_workers=None):
#     """
#     Evaluate a single IQA model on multiple datasets in parallel using threads.
    
#     Args:
#         model          : PyTorch IQA model (pyiqa.create_metric(...))
#         datasets_dict  : dict of {dataset_name: dataset_split}
#         batch_size     : images per batch
#         device         : 'cuda' or 'cpu'
#         max_workers    : max threads, defaults to number of datasets
    
#     Returns:
#         results_dict : dict of {dataset_name: (srcc, plcc, rmse)}
#     """
#     results_dict = {}

#     def eval_wrapper(name, dataset):
#         # Each thread gets its own copy of the model on the same device
#         model_copy = model.__class__.from_pretrained(model.name).to(device).eval() if hasattr(model, 'name') else model
#         srcc, plcc, rmse, _, _ = evaluate_iqa_model(model_copy, dataset, batch_size=batch_size, device=device)
#         return name, (srcc, plcc, rmse)

#     max_workers = max_workers or len(datasets_dict)
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = {executor.submit(eval_wrapper, name, dataset): name for name, dataset in datasets_dict.items()}
#         for future in as_completed(futures):
#             dataset_name, metrics = future.result()
#             results_dict[dataset_name] = metrics
#             print(f"Finished {dataset_name}: SRCC={metrics[0]:.4f}, PLCC={metrics[1]:.4f}, RMSE={metrics[2]:.4f}")

#     return results_dict

In [18]:
# from pyiqa import load_dataset
# import pyiqa

# # Load datasets
# datasets_dict = {
#     'TID2013': load_dataset('tid2013', data_root='./IQA_benchmark_datasets', force_download=False),
#     'LIVE': load_dataset('live', data_root='./IQA_benchmark_datasets', force_download=False),
#     'CSIQ': load_dataset('csiq', data_root='./IQA_benchmark_datasets', force_download=False),
#     'KONIQ10k': load_dataset('koniq10k', data_root='./IQA_benchmark_datasets', force_download=False),
# }

# # Load MANIQA model
# maniqa = pyiqa.create_metric('maniqa-pipal')

# # Evaluate on all datasets in parallel
# results = evaluate_model_on_multiple_datasets_parallel(maniqa, datasets_dict, batch_size=2, device=device)

# # Summary
# print("\n=== Parallel Summary Results ===")
# for dataset_name, (srcc, plcc, rmse) in results.items():
#     print(f"{dataset_name}: SRCC={srcc:.4f}, PLCC={plcc:.4f}, RMSE={rmse:.4f}")


In [42]:
import pyiqa
from pyiqa import load_dataset
# from torch.utils.data import Subset
import numpy as np
import torch

# Resolving dtype issues relating to version compatability
np.float_ = np.float64
np.complex_ = np.complex128

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

dts = {'tid2013': load_dataset('tid2013', data_root='./IQA_benchmark_datasets', force_download=False),
       'live': load_dataset('live', data_root='./IQA_benchmark_datasets', force_download=False),
       'csiq': load_dataset('csiq', data_root='./IQA_benchmark_datasets', force_download=False),
       'koniq10k': load_dataset('koniq10k', data_root='./IQA_benchmark_datasets', force_download=False)}

models = {'paq2piq': pyiqa.create_metric('paq2piq').to(device),
          'brisque': pyiqa.create_metric('brisque').to(device),
          'maniqa': pyiqa.create_metric('maniqa-pipal').to(device),
          'niqe': pyiqa.create_metric('niqe').to(device)}

results = {}
# results = results
print(results.keys())

# limit = 200
for model_name, model in models.items():
    for dataset_name, dataset in dts.items():
        print(f"Evaluating {model_name} on {dataset_name}")
        # dataset = Subset(dataset, range(limit))
        results[(model_name, dataset_name)] = evaluate_iqa_model(model, dataset, batch_size=1, device=device)


Using device: cuda
Loading dataset tid2013 from ./IQA_benchmark_datasets ...
Loading dataset live from ./IQA_benchmark_datasets ...
Loading dataset csiq from ./IQA_benchmark_datasets ...
Loading dataset koniq10k from ./IQA_benchmark_datasets ...
Loading pretrained model PAQ2PIQ from C:\Users\User\.cache\torch\hub\pyiqa\P2P_RoIPoolModel-fit.10.bs.120-ca69882e.pth
Loading pretrained model MANIQA from C:\Users\User\.cache\torch\hub\pyiqa\MANIQA_PIPAL-ae6d356b.pth
dict_keys([])
Evaluating paq2piq on tid2013
Processed 50/3000 images
Processed 100/3000 images
Processed 150/3000 images
Processed 200/3000 images
Processed 250/3000 images
Processed 300/3000 images
Processed 350/3000 images
Processed 400/3000 images
Processed 450/3000 images
Processed 500/3000 images
Processed 550/3000 images
Processed 600/3000 images
Processed 650/3000 images
Processed 700/3000 images
Processed 750/3000 images
Processed 800/3000 images
Processed 850/3000 images
Processed 900/3000 images
Processed 950/3000 image

In [ ]:
for (model_name, dataset_name), results in results.items():
    print(f"Results for {model_name} on {dataset_name}:")
    print(f"  SRCC: {results[0]:.4f}")
    print(f"  PLCC: {results[1]:.4f}")
    print(f"  RMSE: {results[2]:.4f}")

'''
Results for paq2piq on tid2013:
  SRCC: 0.4014
  PLCC: 0.5793
  RMSE: 1.0104
Results for paq2piq on live:
  SRCC: 0.4792
  PLCC: 0.5356
  RMSE: 23.0719
Results for paq2piq on csiq:
  SRCC: 0.5807
  PLCC: 0.6575
  RMSE: 0.1978
Results for paq2piq on koniq10k:
  SRCC: 0.6432
  PLCC: 0.7116
  RMSE: 10.8361
Results for brisque on tid2013:
  SRCC: -0.3814
  PLCC: -0.4581
  RMSE: 9.0881
Results for brisque on live:
  SRCC: -0.9354
  PLCC: -0.9371
  RMSE: 110.2442
Results for brisque on csiq:
  SRCC: -0.6061
  PLCC: -0.7113
  RMSE: 0.8159
Results for brisque on koniq10k:
  SRCC: -0.1312
  PLCC: -0.1533
  RMSE: 118.4804
Results for maniqa on tid2013:
  SRCC: 0.6013
  PLCC: 0.7040
  RMSE: 0.8804
Results for maniqa on live:
  SRCC: 0.8526
  PLCC: 0.8538
  RMSE: 14.2238
Results for maniqa on csiq:
  SRCC: 0.7330
  PLCC: 0.7755
  RMSE: 0.1657
Results for maniqa on koniq10k:
  SRCC: 0.6589
  PLCC: 0.7239
  RMSE: 10.6416
Results for niqe on tid2013:
  SRCC: -0.3119
  PLCC: -0.3986
  RMSE: 9.0752
Results for niqe on live:
  SRCC: -0.9086
  PLCC: -0.9068
  RMSE: 109.6752
Results for niqe on csiq:
  SRCC: -0.6280
  PLCC: -0.7188
  RMSE: 0.8172
Results for niqe on koniq10k:
  SRCC: -0.3828
  PLCC: -0.3985
  RMSE: 118.8872
'''

Results for paq2piq on tid2013:
  SRCC: 0.4014
  PLCC: 0.5793
  RMSE: 1.0104
Results for paq2piq on live:
  SRCC: 0.4792
  PLCC: 0.5356
  RMSE: 23.0719
Results for paq2piq on csiq:
  SRCC: 0.5807
  PLCC: 0.6575
  RMSE: 0.1978
Results for paq2piq on koniq10k:
  SRCC: 0.6432
  PLCC: 0.7116
  RMSE: 10.8361
Results for brisque on tid2013:
  SRCC: -0.3814
  PLCC: -0.4581
  RMSE: 9.0881
Results for brisque on live:
  SRCC: -0.9354
  PLCC: -0.9371
  RMSE: 110.2442
Results for brisque on csiq:
  SRCC: -0.6061
  PLCC: -0.7113
  RMSE: 0.8159
Results for brisque on koniq10k:
  SRCC: -0.1312
  PLCC: -0.1533
  RMSE: 118.4804
Results for maniqa on tid2013:
  SRCC: 0.6013
  PLCC: 0.7040
  RMSE: 0.8804
Results for maniqa on live:
  SRCC: 0.8526
  PLCC: 0.8538
  RMSE: 14.2238
Results for maniqa on csiq:
  SRCC: 0.7330
  PLCC: 0.7755
  RMSE: 0.1657
Results for maniqa on koniq10k:
  SRCC: 0.6589
  PLCC: 0.7239
  RMSE: 10.6416
Results for niqe on tid2013:
  SRCC: -0.3119
  PLCC: -0.3986
  RMSE: 9.0752
Res